# Amazon SageMaker - Tổng quan & Kiến trúc Training & Deployment

## 1. SageMaker là gì?
- **Amazon SageMaker**: Dịch vụ **fully-managed** chính của AWS dành cho **Machine Learning**
- Hỗ trợ **toàn bộ ML lifecycle** (end-to-end):
  - Thu thập & chuẩn bị dữ liệu
  - Training & evaluation model
  - Hyperparameter tuning
  - Deploy model vào production
  - Monitoring & inference
- Hỗ trợ **traditional ML**, **Deep Learning**, và cả **Generative AI** (nhưng GenAI thuần dùng Bedrock tốt hơn)
- SageMaker tồn tại từ trước khi GenAI bùng nổ → tập trung rộng vào ML engineering

> **Quan trọng cho thi MLA-C01**: SageMaker là service **core**, chiếm tỷ lệ lớn ở Domain 2 & Domain 3.

## 2. Kiến trúc Training & Deployment (Conceptual Flow)

### Bottom-up (Quy trình thực tế):

**Training Phase:**
- **Training Code** (container image) → lưu trong **Amazon ECR**
- **Training Data** → lấy từ **Amazon S3** (đã chuẩn bị sẵn)
- Training Job chạy → sinh ra **Model Artifacts** → lưu vào **S3**

**Deployment / Inference Phase:**
- Model Artifacts (S3) + **Inference Code** (container từ ECR)
- Deploy thành **SageMaker Endpoint(s)** (có thể scale nhiều)
- **Client Application** → gọi inference qua **Endpoint**

### Sơ đồ tóm tắt:



**Training Job → Output:**
- Input: Training Image (ECR) + Data (S3)
- Output: Model Artifacts (S3)

## 3. Các cách sử dụng SageMaker

| Cách thực hiện              | Mô tả                                                                 | Phù hợp với |
|-----------------------------|-----------------------------------------------------------------------|-------------|
| **SageMaker Notebook**      | Jupyter Notebook (chạy trên EC2), viết Python code. Có sẵn thư viện (scikit-learn, TensorFlow, PyTorch, Spark...) | Data Scientist, ML Engineer thích code |
| **SageMaker Console (UI)**  | Làm hoàn toàn qua giao diện web, không cần code nhiều. Dùng built-in algorithms | Người mới, nhanh prototype |
| **SageMaker Python SDK**    | Viết code chuyên nghiệp (Estimator, Processor, Pipeline)              | Production, MLOps, automation |

**Đặc điểm Notebook:**
- Tự động spin up/down trên EC2
- Có sẵn kết nối S3
- Nhiều thư viện ML phổ biến được cài sẵn
- Có thể orchestrate toàn bộ: Data Processing → Training → Tuning → Deploy

## 4. Điểm nhấn thi MLA-C01
- SageMaker **không chỉ là nơi train model** mà là **platform quản lý toàn bộ ML workflow**
- Hiểu rõ vai trò:
  - **S3**: Lưu training data & model artifacts
  - **ECR**: Lưu training & inference container images
  - **Endpoint**: Nơi phục vụ prediction ở production (real-time / batch)
- Có thể dùng **built-in algorithms** mà không cần viết code thuật toán từ đầu

**Mẹo note**: 
- Tập trung vào **"Training Job → Model Artifacts → Endpoint"**
- Hiểu cách orchestrate từ Notebook hoặc Console

---

# SageMaker Domain - Kiến trúc & Cấu hình Cơ bản

## 1. SageMaker Domain là gì?
- **SageMaker Domain** là **tổ chức chính (umbrella)** để sử dụng SageMaker Studio.
- **Bắt buộc phải tạo Domain** trước khi dùng bất kỳ tính năng nào của SageMaker.
- Toàn bộ hoạt động trong SageMaker (users, notebooks, apps, data…) đều nằm trong **một Domain**.

> **Quan trọng thi MLA-C01**: Domain là điểm khởi đầu của mọi kiến trúc SageMaker.

## 2. Thành phần chính trong SageMaker Domain

- **EFS Volume** (Elastic File System):
  - Một EFS volume duy nhất được tạo dưới domain.
  - Được chia sẻ cho toàn domain (shared + private directories).

- **User Profiles**:
  - Đại diện cho từng người dùng (user-centric).
  - Mỗi user profile có:
    - Private EFS directory (không gian riêng tư).
    - Personal applications (SageMaker Studio, Canvas…).

- **Shared Resources**:
  - Shared EFS directory (dùng chung giữa các user).
  - Shared Spaces: nơi chia sẻ notebooks, data, kết quả giữa nhiều người.

- **Applications**:
  - SageMaker Studio (IDE chung).
  - Các ứng dụng cá nhân của từng user profile.

## 3. VPC & Network Configuration (Phần quan trọng)

Khi tạo SageMaker Domain, mặc định có **2 VPC**:

| VPC Type                  | Mục đích                                                                 | Quản lý bởi     |
|---------------------------|--------------------------------------------------------------------------|-----------------|
| **Internet-facing VPC**   | Kết nối ra internet (tải package, data từ public, deploy ra ngoài)      | SageMaker quản lý tự động |
| **Customer VPC**          | Truy cập EFS volume (dữ liệu private), traffic nội bộ                    | Bạn phải chỉ định |

**Các thông tin cần cấu hình khi tạo Domain:**
- VPC ID (của bạn)
- Subnets (thường chọn tất cả subnets)
- Security Groups
- VPC-only mode (không dùng internet VPC của SageMaker)

### Hai chế độ VPC phổ biến:
1. **Default mode** (2 VPC): Dễ dùng, có internet access.
2. **VPC-only mode**: Tất cả traffic đi qua VPC của bạn → bảo mật cao hơn, kiểm soát chặt chẽ (khuyến nghị production).

## 4. Tóm tắt SageMaker Domain
- Là **organizational unit** lớn nhất trong SageMaker.
- Bao gồm: Users, User Profiles, Shared Spaces, Applications, EFS, VPC settings.
- Mỗi user có không gian riêng + có thể chia sẻ tài nguyên.
- Network security được quản lý chủ yếu qua **VPC + Subnets + Security Groups**.

**Mẹo ôn thi**:
- Phải tạo Domain trước khi dùng SageMaker Studio.
- Hiểu rõ sự khác biệt giữa **Private directory** và **Shared directory**.
- VPC-only mode thường được dùng trong môi trường production/secure.

---

# SageMaker - Data Preparation → Training → Deployment

## 1. Data Preparation (Data Prep)

**Nguồn dữ liệu chính:**
- **Amazon S3** (phổ biến nhất)
- **FSx for Lustre** (dành cho workload quy mô lớn, hiệu suất cao)
- Các nguồn khác:
  - Amazon Athena
  - Elastic MapReduce (EMR)
  - Amazon Redshift
  - Amazon Keyspaces

**Công cụ xử lý dữ liệu:**
- Tích hợp **Apache Spark** (không chỉ AWS)
- Python libraries có sẵn trong SageMaker Notebook:
  - scikit-learn, NumPy, pandas
- **Processing Container**:
  - Dùng **built-in processing containers** (không cần viết code)
  - Hoặc custom container (từ ECR)

**Quy trình đơn giản:**
1. Copy data từ S3 vào Processing Job
2. Xử lý → Output ra **S3 bucket mới** (đã cleaned & formatted)
3. Dữ liệu sau xử lý sẽ được truyền sang Training Job

> **Lưu ý**: Định dạng dữ liệu nên phù hợp với algorithm (RecordIO, protobuf, columnar…).

## 2. Training Model

**Tạo Training Job:**
- Input: Đường dẫn S3 chứa dữ liệu đã xử lý
- Chọn **compute resources** (instance type & số lượng) → chi phí cao ở bước này
- Training code: Lưu trong **Amazon ECR** (container)

**Các lựa chọn Training Code:**
- **Built-in Algorithms** (SageMaker cung cấp sẵn)
- **Framework Containers**:
  - TensorFlow, PyTorch, MXNet
  - Spark ML
  - scikit-learn
  - Reinforcement Learning (RL)
  - XGBoost
  - Hugging Face (LLM, SLM, GenAI models)
  - Chainer
- **Custom Docker Image** (bất kỳ thứ gì bạn muốn)
- Algorithms mua từ **AWS Marketplace**

**Output của Training:**
- **Model Artifacts** → lưu vào **S3 bucket** chỉ định

## 3. Deployment (Triển khai Model)

**2 cách triển khai chính:**

| Loại                  | Tên tính năng                  | Mô tả                                                                 | Khi nào dùng?                  |
|-----------------------|--------------------------------|-----------------------------------------------------------------------|--------------------------------|
| **Real-time**         | SageMaker Endpoint             | Persistent endpoint, scale tự động, phục vụ prediction ngay lập tức   | Cần prediction real-time       |
| **Batch**             | SageMaker Batch Transform      | Xử lý batch lớn một lần, không cần endpoint                           | Dự đoán hàng loạt, không realtime |

**Tính năng nâng cao khác:**
- **Inference Pipelines**: Orchestrate chuỗi xử lý phức tạp trước/sau inference
- **SageMaker Neo**: Deploy model ra **edge devices** (thiết bị biên, ít kết nối internet)
- **Elastic Inference**: Tăng tốc inference (gắn thêm accelerator)
- **Automatic Scaling**: Tự động scale số endpoint theo traffic
- **Shadow Testing**: Test model mới song song với model cũ (không ảnh hưởng production)
- **Rollback**: Dễ dàng rollback nếu model mới không đạt yêu cầu

**Mẹo thi MLA-C01**:
- Hiểu rõ flow: **S3 (raw) → Processing Job → S3 (processed) → Training Job → Model Artifacts (S3) → Endpoint / Batch Transform**
- SageMaker hỗ trợ **rất nhiều framework** (built-in + custom container)
- Deployment không chỉ có real-time mà còn batch, edge, auto-scale, shadow test

---

# SageMaker Ground Truth - Dịch vụ Gán nhãn dữ liệu

## 1. SageMaker Ground Truth là gì?
- Dịch vụ của AWS giúp **sử dụng con người để gán nhãn (label) dữ liệu** cho mục đích huấn luyện ML.
- Thường dùng khi dữ liệu thiếu **labels** (nhãn) hoặc thiếu **features**.
- Phổ biến nhất trong **Computer Vision** (ví dụ: gán nhãn ảnh là bóng rổ hay bóng đá, tìm chim trong ảnh…).

> **Vị trí trong Feature Engineering**: Vì nó giúp tạo/gán nhãn và tạo thêm features cho mô hình.

## 2. Cách Ground Truth hoạt động (Thông minh & Tiết kiệm)

- Ban đầu: Gửi task cho **con người** để gán nhãn.
- Trong quá trình: Ground Truth **tự động xây dựng một model** dựa trên nhãn từ con người.
- Sau đó: Chỉ gửi những case **khó / mơ hồ** cho con người. Những case dễ thì model tự dự đoán.
- **Lợi ích**: Giảm chi phí labeling lên đến **70%**.

## 3. Các loại Workforce (Đội ngũ gán nhãn)

| Loại Workforce                  | Mô tả                                                                 | Phù hợp khi |
|--------------------------------|-----------------------------------------------------------------------|-------------|
| **Amazon Mechanical Turk**     | Đội ngũ lớn trên toàn thế giới, chi phí thấp                          | Dự án thông thường, ngân sách hạn chế |
| **Private Workforce**          | Đội ngũ nội bộ của công ty bạn                                       | Dữ liệu nhạy cảm, bảo mật cao |
| **Vendor Workforce**           | Công ty chuyên labeling chuyên nghiệp                                 | Cần chất lượng cao, chuyên sâu |

## 4. Ground Truth Plus (Dịch vụ cao cấp)
- AWS làm **toàn bộ** thay bạn (turnkey solution).
- Đội ngũ chuyên gia AWS thiết lập workflow, quản lý labeler, theo dõi tiến độ.
- Bạn chỉ cần điền form mô tả yêu cầu → AWS liên hệ báo giá (không công khai, thường đắt).
- Theo dõi tiến độ qua **Ground Truth Plus Project Portal** (có dashboard, charts).
- Kết quả cuối cùng: Nhận dữ liệu đã gán nhãn từ **S3**.

## 5. Các cách thay thế / Bổ sung (Không dùng người)

- **Amazon Rekognition**: Dùng pre-trained model để tự động gán nhãn ảnh/video (object detection, classification…).
- **Amazon Comprehend**: Phân tích text → tạo features như topic, sentiment, entity… (rất hữu ích cho NLP).
- Bất kỳ **pre-trained model** hoặc **unsupervised learning** nào cũng có thể dùng để sinh thêm labels hoặc features.

> **Tóm tắt**: Ground Truth dùng để tạo **ground truth labels** khi máy không làm được. Kết hợp với Rekognition/Comprehend để giảm chi phí và tăng tốc độ feature engineering.

## 6. Điểm nhấn thi MLA-C01
- Ground Truth thuộc phần **Data Preparation / Feature Engineering**.
- Hiểu rõ cơ chế **Active Learning** (model tự học và chỉ gửi case khó cho người).
- Biết sự khác biệt giữa Ground Truth thông thường và **Ground Truth Plus**.
- Có thể kết hợp với AI services (Rekognition, Comprehend) để tự động hóa labeling.

---

# Amazon Mechanical Turk (MTurk)

## 1. Mechanical Turk là gì?
- Dịch vụ **crowdsourcing marketplace** của AWS.
- Cho phép bạn tiếp cận **đội ngũ lao động ảo phân tán** (virtual workforce) trên toàn thế giới.
- Con người sẽ thực hiện các **task đơn giản** với chi phí rất rẻ.

**Nguồn gốc tên gọi:**
- Lấy cảm hứng từ “Mechanical Turk” năm 1770: Một robot chơi cờ vua giả tạo (thực ra có người ẩn bên trong điều khiển).

## 2. Cách hoạt động
- Bạn tạo **tasks** (công việc) trên Mechanical Turk.
- Con người khắp nơi trên thế giới nhận làm và hoàn thành task.
- Bạn **quyết định giá thưởng** (reward) cho mỗi task.
- Ví dụ: 
  - 10 triệu ảnh, reward 0.1 USD/ảnh → tổng chi phí 1 triệu USD.

**Giao diện cho Worker:**
- Worker thấy danh sách jobs + reward.
- Họ chấp nhận job → làm → nhận tiền.

## 3. Use Cases phổ biến
- **Image classification / labeling** (gán nhãn ảnh)
- Data collection
- Data validation / review
- Business process outsourcing (nhập liệu, điền form, kiểm tra Excel…)
- Review recommendations

## 4. Vai trò trong Machine Learning & AI
- Dùng để **tạo nhãn dữ liệu (labeling)** cho tập huấn luyện.
- Hỗ trợ **feature engineering** khi cần dữ liệu do con người tạo ra.
- Tích hợp sâu với:
  - **SageMaker Ground Truth**
  - **Amazon A2I** (Augmented AI)

## 5. Điểm nhấn thi MLA-C01
- Mechanical Turk là một loại **Workforce** trong SageMaker Ground Truth.
- Phù hợp khi cần **lượng dữ liệu lớn** với chi phí thấp.
- Ưu điểm: Nhanh, rẻ, quy mô lớn.
- Nhược điểm: Chất lượng phụ thuộc vào reward và task design; không phù hợp với dữ liệu nhạy cảm.

**So sánh nhanh với Ground Truth:**
- Mechanical Turk = Workforce công khai, chi phí thấp.
- Private Workforce = Đội ngũ nội bộ (bảo mật cao hơn).
- Vendor Workforce = Đội chuyên nghiệp.

---

# SageMaker Data Wrangler - Công cụ ETL cho Machine Learning

## 1. SageMaker Data Wrangler là gì?
- Công cụ **ETL (Extract - Transform - Load)** được tích hợp sẵn trong **SageMaker Studio**.
- Được thiết kế **chuyên biệt cho Machine Learning pipelines** (khác với Glue DataBrew).
- Cho phép import, visualize, transform dữ liệu một cách trực quan (low-code / no-code).

> **Quan trọng thi MLA-C01**: Đây là công cụ quan trọng nhất trong Data Preparation cho ML.

## 2. Chức năng chính

- **Import Data** từ nhiều nguồn:
  - S3 (CSV, Parquet…)
  - Lake Formation
  - Athena
  - SageMaker Feature Store
  - Redshift
  - JDBC (Salesforce, Databricks, …)

- **Visualize Data**:
  - Xem phân bố dữ liệu, phát hiện outliers
  - Kiểm tra data types, thay đổi nếu cần
  - Sanity check trước khi đưa vào model

- **Transform Data**:
  - Hơn **300 built-in transformations**
  - Hỗ trợ custom code: **pandas**, **PySpark**, **PySpark SQL**
  - Ví dụ phổ biến: One-hot encoding (Encode Categorical), Normalization, Handling missing values…

- **Quick Model** (Tính năng nổi bật):
  - Train nhanh một mô hình đơn giản trên dữ liệu đã transform
  - Đánh giá kết quả → giúp chọn transformation tối ưu cho model

## 3. Kiến trúc & Cách hoạt động
```
Data Sources
↓ (S3, Athena, Feature Store, Redshift, JDBC…)
SageMaker Data Wrangler (Visual Flow)
↓
Export → Python Code (Jupyter Notebook)
↓
Sử dụng trong:
→ SageMaker Processing
→ SageMaker Pipelines
→ SageMaker Feature Store
```

**Lưu ý quan trọng**:
- Data Wrangler **không thực hiện transformation** trong pipeline.
- Nó chỉ **tạo code** (code generation tool) để bạn đưa vào notebook hoặc pipeline.

## 4. Export & Sử dụng
- Export ra **Jupyter Notebook** chứa toàn bộ Python code cho ETL flow.
- Code này có thể chạy lại nhiều lần, tích hợp vào SageMaker Pipelines hoặc Processing Job.

## 5. Troubleshooting phổ biến
- **Permission error**: 
  - Cần attach IAM role phù hợp cho SageMaker Studio user.
  - Data source phải cho phép SageMaker truy cập (SageMakerFullAccess).
- **Resource limit error** ("instance type is not available"):
  - Không phải hết instance, mà là quota chưa đủ.
  - Giải pháp: Vào **Service Quotas** → Amazon SageMaker → Studio Kernel Gateway apps → Request tăng quota (ví dụ: ML.m5.4xlarge).

## 6. Điểm nhấn thi MLA-C01
- Data Wrangler thuộc **Domain 1: Data Preparation**.
- Ưu điểm: Trực quan, nhanh prototype, có Quick Model để test transformation.
- Nhược điểm: Chỉ sinh code, không phải công cụ chạy production trực tiếp.
- Thường được dùng kết hợp với **SageMaker Processing** và **SageMaker Pipelines**.

---

# SageMaker Model Monitor - Giám sát Model trong Production

## 1. Tại sao cần Model Monitor?
- Model sau khi deploy vào production **không duy trì hiệu suất mãi mãi**.
- Dữ liệu thực tế có thể thay đổi theo thời gian → gây **Data Drift**, **Model Drift**, bias…
- SageMaker Model Monitor giúp **tự động phát hiện** và **cảnh báo** các vấn đề này.

## 2. Chức năng chính của SageMaker Model Monitor

- **Data Drift**: Phát hiện sự thay đổi phân bố của dữ liệu đầu vào.
- **Data Quality**: Phát hiện missing data, outliers, anomalies, new features.
- **Model Quality**: Giám sát độ chính xác (accuracy) của model theo thời gian.
- **Bias Drift**: Phát hiện sự thay đổi bias (phân biệt đối xử) theo thời gian.
- **Feature Attribution Drift**: Phát hiện sự thay đổi mức độ quan trọng của từng feature.

**Tích hợp với SageMaker Clarify**:
- Clarify giúp phát hiện **bias** trong model.
- Giải thích model (feature importance).
- Hỗ trợ nhiều bias metrics (ví dụ: Kullback-Leibler divergence).

## 3. Kiến trúc & Cách hoạt động

- **Monitoring Schedule**: Thiết lập lịch chạy monitoring job (có thể chạy định kỳ).
- **Baseline**: Định nghĩa tiêu chuẩn chất lượng ban đầu để so sánh.
- **Output**:
  - Metrics được lưu trong **S3** (an toàn).
  - Metrics được gửi đến **CloudWatch**.
- Từ CloudWatch → tạo **Alarms** và **Notifications** → trigger hành động (retrain model, audit data…).

**Tích hợp Ground Truth**:
- So sánh prediction của model với **nhãn từ con người** (ground truth labels) để đo model quality.

## 4. Visualization & Reporting
- Visualize trực tiếp trong **SageMaker Studio**.
- Hỗ trợ tích hợp với:
  - TensorBoard
  - Amazon QuickSight
  - Tableau

## 5. Các loại Monitoring

| Monitoring Type              | Mô tả                                                                 |
|-----------------------------|-----------------------------------------------------------------------|
| **Data Quality Drift**      | Giám sát chất lượng dữ liệu so với baseline                           |
| **Model Quality Drift**     | Giám sát độ chính xác của model (so với ground truth)                 |
| **Bias Drift**              | Giám sát sự thay đổi bias theo thời gian                              |
| **Feature Attribution Drift** | Giám sát sự thay đổi mức độ ảnh hưởng của từng feature                |

## 6. Điểm nhấn thi MLA-C01 (Domain 4)
- Model Monitor thuộc phần **ML Solution Monitoring & Maintenance**.
- Không cần viết code nhiều → cấu hình qua SageMaker Studio.
- Luôn kết hợp với **CloudWatch Alarms** để nhận thông báo.
- Thường dùng cùng **SageMaker Clarify** để monitor bias.
- Hành động sau khi nhận alert: Retraining model hoặc kiểm tra dữ liệu.

**Mẹo ôn**:
- Model Monitor = Giám sát **sự thay đổi theo thời gian** (drift).
- Clarify = Giải thích model + phát hiện bias ban đầu và drift.

---

# SageMaker Clarify - Giải thích Model & Phát hiện Bias

## 1. Mục đích của SageMaker Clarify
- Giải thích hành vi của model (model explainability).
- Phát hiện **bias** (phân biệt đối xử) trong data và model.
- Phân tích mức độ ảnh hưởng của từng feature đến dự đoán.
- Hỗ trợ **debugging** và cải thiện chất lượng model.

> **Quan trọng thi MLA-C01**: Clarify là công cụ chính trong Domain 4 (Monitoring & Maintenance), thường kết hợp với Model Monitor.

## 2. Partial Dependence Plot (PDP)

- **PDP** cho thấy mối quan hệ giữa **một feature** và **kết quả dự đoán** của model.
- Trục X: Giá trị của feature (được bucket hóa).
- Trục Y: Giá trị dự đoán trung bình của model.
- Giúp phát hiện:
  - Feature nào quan trọng.
  - Điểm "tipping point" (ví dụ: tuổi > 50 thì dự đoán không thay đổi).
  - Vấn đề imbalance dữ liệu (ít dữ liệu ở một số khoảng giá trị).

**Lợi ích**: 
- Dễ hình dung tác động của feature.
- Có thể lấy raw data distribution cho từng bucket.

## 3. Shapley Values & SHAP (SHapley Additive exPlanations)

- **Shapley Values**: Kỹ thuật từ lý thuyết trò chơi, đo lường mức độ đóng góp của từng feature vào output của model.
- **SHAP**: Phương pháp xấp xỉ (approximation) hiệu quả của Shapley Values → SageMaker Clarify dùng SHAP dưới hood.
- Ý tưởng: 
  - Loại bỏ từng feature (hoặc tổ hợp) và đo sự thay đổi trong dự đoán.
  - Tính trung bình đóng góp của từng feature.

**Ưu điểm của SHAP**:
- Giải thích rõ ràng từng prediction.
- Phát hiện bias tiềm ẩn.
- Chỉ ra feature nào đang "lái" model theo hướng không mong muốn.

## 4. Asymmetric Shapley Values (cho Time Series)
- Dùng riêng cho **dữ liệu chuỗi thời gian** (time series).
- Đo lường tác động của features **tại từng time step** đến forecast/prediction.
- Phức tạp hơn SHAP thông thường vì có yếu tố thời gian.

## 5. Những gì Clarify cung cấp
- **Graphical plots**: Partial Dependence Plot (PDP).
- **Raw data**: Data distributions, class imbalance.
- **Feature importance & attribution**.
- **Bias metrics** (nhiều loại, không cần nhớ chi tiết).
- Tích hợp với **Model Monitor** để theo dõi **Bias Drift** theo thời gian.

## 6. Điểm nhấn thi MLA-C01
- Clarify = **Explainability + Bias Detection**.
- PDP: Hiển thị dependence giữa feature và prediction.
- SHAP: Kỹ thuật chính để tính feature impact.
- Asymmetric Shapley: Dùng cho time series data.
- Thường được dùng cùng Model Monitor để giám sát bias drift và feature attribution drift.

**Mẹo ôn**: 
- Clarify giúp trả lời câu hỏi “Tại sao model dự đoán như vậy?” và “Model có đang bias không?”.

---

# SageMaker Feature Store - Kho lưu trữ Features cho ML

## 1. Feature là gì?
- **Feature**: Thuộc tính / cột dữ liệu dùng để huấn luyện model.
- Ví dụ: Dự đoán đảng phái chính trị → Features = age, income, address…  
  Label = political party.

## 2. Tại sao cần SageMaker Feature Store?
- Cần truy cập **nhanh, an toàn, quy mô lớn** vào features để train & inference.
- Tránh lặp lại (duplicate) features giữa nhiều model.
- Dễ dàng **chia sẻ features** giữa các team và các model khác nhau.
- Hỗ trợ cả **streaming** và **batch** data.

## 3. Kiến trúc SageMaker Feature Store

- **Feature Group**: Nhóm các features liên quan lại với nhau.
  - Mỗi Feature Group chứa:
    - Record Identifier (khóa chính)
    - Feature Names
    - Event Time (thời gian sự kiện)

- **Hai loại Store**:
  | Loại Store       | Mục đích                              | Công nghệ                  | Cách truy cập                  |
  |------------------|---------------------------------------|----------------------------|--------------------------------|
  | **Online Store** | Low-latency inference (real-time)     | Managed service            | GetRecord API                  |
  | **Offline Store**| Batch training, querying lớn          | S3 + Glue Data Catalog     | Athena, Data Wrangler, Spark…  |

## 4. Nguồn dữ liệu vào Feature Store
- SageMaker Studio / Notebooks
- SageMaker Pipelines / Step Functions / Airflow
- SageMaker Processing / Data Wrangler
- AWS Glue / Glue DataBrew
- Amazon EMR / Spark
- Streaming: Kinesis, MSK (Kafka), Apache Spark Streaming

**Cách đưa dữ liệu vào**:
- **Streaming**: Dùng `PutRecord` API
- **Batch**: Dump trực tiếp vào Offline Store (S3)

## 5. Tính năng nổi bật
- Tự động tạo **Glue Data Catalog** cho Offline Store → dễ query bằng Athena.
- **Security**:
  - Mã hóa tự động (at rest & in transit)
  - Hỗ trợ KMS Customer Managed Key
  - Fine-grained access control bằng IAM
  - Hỗ trợ AWS PrivateLink
- Hỗ trợ cả **streaming** (real-time) và **batch** use cases.

## 6. Điểm nhấn thi MLA-C01 (Domain 1 & 2)
- Feature Store thuộc phần **Data Preparation** và **Model Development**.
- Phân biệt rõ **Online Store** (inference nhanh) vs **Offline Store** (training & analytics).
- Hiểu cách Feature Store giúp **tổ chức và chia sẻ features** giữa nhiều model.
- Thường được dùng kết hợp với Data Wrangler, SageMaker Pipelines và Model Monitor.

**Tóm tắt một câu**:
SageMaker Feature Store là kho lưu trữ trung tâm, an toàn, hỗ trợ streaming + batch, giúp quản lý và chia sẻ features hiệu quả cho toàn bộ ML lifecycle.

---

# SageMaker Canvas - No-Code / Low-Code Machine Learning

## 1. SageMaker Canvas là gì?
- **No-code / Low-code** ML environment trong SageMaker.
- Được thiết kế chủ yếu cho **Business Analysts** và người không biết code ML.
- Cho phép xây dựng model chỉ bằng giao diện kéo-thả và click.

> **Quan trọng thi MLA-C01**: Canvas sẽ xuất hiện 1–2 câu hỏi. Bạn cần biết rõ đối tượng sử dụng và khả năng của nó.

## 2. Chức năng chính

- **AutoML**: 
  - Tự động xây dựng model từ file CSV.
  - Tự động chọn thuật toán tốt nhất.
  - Hỗ trợ **Classification** và **Regression**.

- **Data Preparation** (tự động):
  - Xử lý missing values.
  - Xử lý outliers.
  - Xóa duplicate rows.
  - Join nhiều dataset lại với nhau.

- **Generative AI Support** (mới):
  - Sử dụng Foundation Models từ **Amazon Bedrock**.
  - Sử dụng models từ **SageMaker JumpStart**.
  - **Fine-tuning** model với dữ liệu riêng ngay trong Canvas (không cần code).

## 3. Workflow cơ bản
1. Upload dữ liệu (CSV hoặc từ nhiều nguồn).
2. Chọn cột cần dự đoán (target column).
3. Canvas tự động:
   - Làm sạch dữ liệu.
   - Train model bằng AutoML.
   - Đánh giá và đưa ra predictions.
4. (Tùy chọn) Export model + dữ liệu sang **SageMaker Studio** để ML Engineer tiếp tục phát triển.

## 4. Tích hợp & Export
- Có thể **share** datasets và models với SageMaker Studio.
- Export model để dùng tiếp trong SageMaker (training job, endpoint, pipeline…).
- Hỗ trợ cả **traditional ML** và **Generative AI** (fine-tune FM).

## 5. Điểm nhấn thi MLA-C01
- Canvas = **ML cho người không phải ML Engineer**.
- Dùng **AutoML** để tự động hóa toàn bộ quá trình.
- Hỗ trợ **Generative AI + Fine-tuning** mà không cần code.
- Có thể export sang Studio → thể hiện sự chuyển giao giữa business user và ML Engineer.
- Thuộc Domain 2: ML Model Development.

**Tóm tắt một câu**:
SageMaker Canvas là giao diện no-code giúp business analyst tự xây dựng model ML và fine-tune GenAI models mà không cần biết lập trình.

---

# AWS Glue - ETL & Data Catalog cho Data Lake

## 1. AWS Glue là gì?
- Dịch vụ **serverless** của AWS chuyên **ETL** (Extract - Transform - Load).
- Vai trò chính:
  - Tạo và quản lý **metadata** cho Data Lake.
  - Khám phá schema tự động từ dữ liệu không cấu trúc trong S3.
  - Thực hiện ETL jobs bằng **Apache Spark** (không cần quản lý cluster).

> **Quan trọng thi MLA-C01**: Glue chiếm tỷ lệ lớn trong Domain 1 (Data Preparation). Đây là service kết nối Data Lake với các công cụ phân tích.

## 2. Thành phần chính của AWS Glue

- **Glue Data Catalog**:
  - Kho metadata trung tâm (central metadata repository).
  - Lưu table definitions (schema, column names, data types, location…).
  - Cho phép query dữ liệu S3 như bảng SQL.

- **Glue Crawler**:
  - Tự động quét dữ liệu trong S3.
  - Tự động suy luận (infer) schema.
  - Có thể chạy theo lịch hoặc on-demand.
  - Populate (điền) dữ liệu vào Glue Data Catalog.

- **Glue ETL Jobs**:
  - Thực hiện transform dữ liệu.
  - Dùng **Apache Spark** dưới hood (serverless).
  - Trigger: Schedule, Event-driven, On-demand.

## 3. Glue hoạt động với S3 như thế nào?
- Dữ liệu **vẫn nằm nguyên** trong S3 (không duplicate).
- Glue chỉ cung cấp **schema** (table definition) để các công cụ khác query.
- Các công cụ có thể dùng Glue Catalog:
  - Amazon Athena
  - Amazon Redshift Spectrum
  - Amazon EMR (Hive, Spark)
  - SageMaker Data Wrangler, v.v.

## 4. Partitions trong S3 (Rất quan trọng cho thi)

- Glue Crawler tự động trích xuất **partitions** dựa trên cách tổ chức thư mục trong S3.
- **Partitioning strategy** ảnh hưởng rất lớn đến hiệu suất query.

**Ví dụ tốt nhất**:
- Dữ liệu sensor từ thiết bị, gửi mỗi giờ.

**Cách tổ chức S3 khuyến nghị:**

| Trường hợp query chính          | Cấu trúc thư mục S3 khuyến nghị                  | Lý do |
|--------------------------------|--------------------------------------------------|-------|
| Query theo thời gian           | `year=2025/month=04/day=10/device=001/`         | Giảm dữ liệu quét khi query theo ngày/tháng |
| Query theo device              | `device=001/year=2025/month=04/day=10/`         | Giảm dữ liệu quét khi query theo từng thiết bị |

**Mẹo thi**: 
- Luôn đặt partition theo **thuộc tính hay được filter nhất** lên đầu (top level).
- Partitioning giúp Glue và Athena/Redshift chỉ quét dữ liệu cần thiết → tiết kiệm chi phí và thời gian.

## 5. Điểm nhấn thi MLA-C01
- Glue = **Cầu nối** giữa Data Lake (S3) và các công cụ phân tích SQL.
- Glue Crawler + Data Catalog là phần cốt lõi.
- Glue ETL dùng Spark serverless.
- Hiểu rõ **partitioning strategy** trong S3 để tối ưu hiệu suất.
- Glue thường được dùng cùng: Athena, SageMaker Data Wrangler, SageMaker Feature Store, SageMaker Pipelines.

**Tóm tắt một câu**:
AWS Glue là dịch vụ serverless giúp khám phá schema, tạo Data Catalog và thực hiện ETL trên Data Lake, biến dữ liệu thô trong S3 thành dữ liệu có thể query bằng SQL.

---